Imports, paths, and configuration

In [12]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import CalibrationDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Project folders:
# Expected structure:
# PPD_Risk_Prediction/
#   dataset/post natal data.csv
#   notebooks/04_model_tuning.ipynb
#   models/
#   outputs/metrics/
#   outputs/graphs/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
GRAPHS_DIR = OUTPUT_DIR / "graphs"

for folder in [MODEL_DIR, METRICS_DIR, GRAPHS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = [
    DATA_DIR / "post natal data.csv",
    DATA_DIR / "raw" / "post natal data.csv",
]
DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Put 'post natal data.csv' in either "
        "'dataset/' or 'dataset/raw/' inside the project folder. "
        f"Checked: {DATA_PATH_CANDIDATES}"
    )

TARGET_COLUMN = "Feeling anxious"
TARGET_MAPPING = {"no": 0, "yes": 1}

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Target:", TARGET_COLUMN)

Project root: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction
Dataset: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction\dataset\raw\post natal data.csv
Target: Feeling anxious


Load and inspect the dataset

In [19]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

print("Original shape:", df.shape)
display(df.head())
display(df.info())

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' not found. Available columns: {df.columns.tolist()}"
    )

# Clean whitespace and common blank/missing text representations.
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace(
        {"": pd.NA, "nan": pd.NA, "None": pd.NA, "NA": pd.NA}
    )

target_text = df[TARGET_COLUMN].astype("string").str.strip().str.lower()
y_mapped = target_text.map(TARGET_MAPPING)

unknown_target_values = sorted(target_text[y_mapped.isna()].dropna().unique().tolist())
if unknown_target_values:
    print("Target values excluded because they are not Yes/No:", unknown_target_values)

df = df.loc[y_mapped.notna()].copy()
df[TARGET_COLUMN] = y_mapped.loc[y_mapped.notna()].astype(int)

# Remove rows with no target and remove timestamp from predictors.
# Separate features and target
X = df.drop(
    columns=[TARGET_COLUMN, "Timestamp"],
    errors="ignore"
).copy()

y = df[TARGET_COLUMN].copy()

# Convert pandas missing values (pd.NA) into NumPy missing values (np.nan)
# This improves compatibility with Scikit-learn.
X = X.astype(object)
X = X.where(pd.notna(X), np.nan)

# Convert all non-missing feature values into clean strings
for column in X.columns:
    X[column] = X[column].apply(
        lambda value: str(value).strip() if pd.notna(value) else np.nan
    )

print("Feature data types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isnull().sum())

print("\nFeatures prepared successfully!")

if y.nunique() != 2:
    raise ValueError(f"Expected two target classes after mapping; found {y.nunique()}.")

print("Cleaned feature shape:", X.shape)
print("Target distribution:")
display(y.value_counts().rename(index={0: "No (0)", 1: "Yes (1)"}))

Original shape: (1503, 11)


,Timestamp,Age,Feeling sad or Tearful,Irritable towards baby & partner,Trouble sleeping at night,Problems concentrating or making decision,Overeating or loss of appetite,Feeling anxious,Feeling of guilt,Problems of bonding with baby,Suicide attempt
0,6/14/2022 20:02,35-40,Yes,Yes,Two or more days a week,Yes,Yes,Yes,No,Yes,Yes
1,6/14/2022 20:03,40-45,Yes,No,No,Yes,Yes,No,Yes,Yes,No
2,6/14/2022 20:04,35-40,Yes,No,Yes,Yes,Yes,Yes,No,Sometimes,No
3,6/14/2022 20:05,35-40,Yes,Yes,Yes,Yes,No,Yes,Maybe,No,No
4,6/14/2022 20:06,40-45,Yes,No,Two or more days a week,Yes,No,Yes,No,Yes,No


<class 'pandas.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 11 columns):
 #   Column                                     Non-Null Count  Dtype
---  ------                                     --------------  -----
 0   Timestamp                                  1503 non-null   str  
 1   Age                                        1503 non-null   str  
 2   Feeling sad or Tearful                     1503 non-null   str  
 3   Irritable towards baby & partner           1497 non-null   str  
 4   Trouble sleeping at night                  1503 non-null   str  
 5   Problems concentrating or making decision  1491 non-null   str  
 6   Overeating or loss of appetite             1503 non-null   str  
 7   Feeling anxious                            1503 non-null   str  
 8   Feeling of guilt                           1494 non-null   str  
 9   Problems of bonding with baby              1503 non-null   str  
 10  Suicide attempt                            1503 non-null   

None

Feature data types:
Age                                          str
Feeling sad or Tearful                       str
Irritable towards baby & partner             str
Trouble sleeping at night                    str
Problems concentrating or making decision    str
Overeating or loss of appetite               str
Feeling of guilt                             str
Problems of bonding with baby                str
Suicide attempt                              str
dtype: object

Missing values:
Age                                           0
Feeling sad or Tearful                        0
Irritable towards baby & partner              6
Trouble sleeping at night                     0
Problems concentrating or making decision    12
Overeating or loss of appetite                0
Feeling of guilt                              9
Problems of bonding with baby                 0
Suicide attempt                               0
dtype: int64

Features prepared successfully!
Cleaned feature shape: (1503, 

Feeling anxious
Yes (1)    980
No (0)     523
Name: count, dtype: int64

Create and save the train/test split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

evaluation_data = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_columns": X.columns.tolist(),
    "target_column": TARGET_COLUMN,
    "target_mapping": TARGET_MAPPING,
    "random_state": RANDOM_STATE,
}
joblib.dump(evaluation_data, MODEL_DIR / "evaluation_data.joblib")

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Saved:", MODEL_DIR / "evaluation_data.joblib")

Training rows: 1202
Testing rows: 301
Saved: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction\models\evaluation_data.joblib


Build preprocessing pipeline

In [21]:
categorical_features = X_train.columns.tolist()

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_preprocessor, categorical_features)
    ],
    remainder="drop"
)

print("Categorical features:", categorical_features)

Categorical features: ['Age', 'Feeling sad or Tearful', 'Irritable towards baby & partner', 'Trouble sleeping at night', 'Problems concentrating or making decision', 'Overeating or loss of appetite', 'Feeling of guilt', 'Problems of bonding with baby', 'Suicide attempt']


Define models and tuning spaces

In [22]:
# Optional model libraries: install them in the same Python environment as this notebook.
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("XGBoost is missing. Run: python -m pip install xgboost") from exc

try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError("LightGBM is missing. Run: python -m pip install lightgbm") from exc

class_counts = y_train.value_counts()
negative_count = int(class_counts.get(0, 0))
positive_count = int(class_counts.get(1, 0))
if positive_count == 0 or negative_count == 0:
    raise ValueError("Both target classes must be present in the training set.")

scale_pos_weight = negative_count / positive_count

models_and_spaces = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        {
            "classifier__C": [0.01, 0.1, 1.0, 10.0],
            "classifier__solver": ["liblinear", "lbfgs"]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "classifier__n_estimators": [100, 200, 300],
            "classifier__max_depth": [None, 5, 10, 20],
            "classifier__min_samples_split": [2, 5, 10],
            "classifier__min_samples_leaf": [1, 2, 4]
        }
    ),
    "SVM": (
        SVC(
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        {
            "classifier__C": [0.1, 1, 10],
            "classifier__kernel": ["linear", "rbf"],
            "classifier__gamma": ["scale", "auto"]
        }
    ),
    "XGBoost": (
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0
        ),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [2, 3, 5],
            "classifier__learning_rate": [0.03, 0.1],
            "classifier__subsample": [0.8, 1.0]
        }
    ),
    "LightGBM": (
        LGBMClassifier(
            objective="binary",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=-1
        ),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [-1, 3, 5],
            "classifier__learning_rate": [0.03, 0.1],
            "classifier__num_leaves": [7, 15, 31]
        }
    )
}

N_ITER = 5
N_SPLITS = min(5, int(y_train.value_counts().min()))
if N_SPLITS < 2:
    raise ValueError("Not enough samples in the minority class for stratified cross-validation.")

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Models:", list(models_and_spaces.keys()))
print("CV folds:", N_SPLITS)
print("Randomized search iterations per model:", N_ITER)

Models: ['Logistic Regression', 'Random Forest', 'SVM', 'XGBoost', 'LightGBM']
CV folds: 5
Randomized search iterations per model: 5


Tune all models

In [23]:
tuned_models = {}
tuning_rows = []

for model_name, (estimator, parameter_space) in models_and_spaces.items():
    print(f"\nTuning: {model_name}")

    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])

    search = RandomizedSearchCV(
        estimator=model_pipeline,
        param_distributions=parameter_space,
        n_iter=N_ITER,
        scoring="average_precision",
        cv=cv,
        refit=True,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=1,
        error_score="raise"
    )

    search.fit(X_train, y_train)

    tuned_models[model_name] = search.best_estimator_
    tuning_rows.append({
        "Model": model_name,
        "Best CV PR-AUC": search.best_score_,
        "Best Parameters": json.dumps(search.best_params_, default=str)
    })

    print("Best CV PR-AUC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    "Best CV PR-AUC", ascending=False
)
display(tuning_results)

joblib.dump(tuned_models, MODEL_DIR / "tuned_models.joblib")
tuning_results.to_csv(METRICS_DIR / "hyperparameter_tuning_results.csv", index=False)

with open(MODEL_DIR / "target_configuration.json", "w", encoding="utf-8") as f:
    json.dump({
        "target_column": TARGET_COLUMN,
        "target_mapping": TARGET_MAPPING,
        "feature_columns": X.columns.tolist(),
        "warning": "Feeling anxious is a symptom proxy, not a validated PPD diagnosis."
    }, f, indent=4)

print("\nSaved tuned models:", MODEL_DIR / "tuned_models.joblib")
print("Saved tuning results:", METRICS_DIR / "hyperparameter_tuning_results.csv")


Tuning: Logistic Regression
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV PR-AUC: 0.9498
Best parameters: {'classifier__solver': 'liblinear', 'classifier__C': 0.1}

Tuning: Random Forest
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV PR-AUC: 0.9986
Best parameters: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 20}

Tuning: SVM
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV PR-AUC: 0.9939
Best parameters: {'classifier__kernel': 'rbf', 'classifier__gamma': 'scale', 'classifier__C': 10}

Tuning: XGBoost
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV PR-AUC: 0.9911
Best parameters: {'classifier__subsample': 1.0, 'classifier__n_estimators': 200, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.03}

Tuning: LightGBM
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV PR-AUC: 0.9921
Best parameters: 

,Model,Best CV PR-AUC,Best Parameters
1,Random Forest,0.998583,"{""classifier__n_estimators"": 300, ""classifier_..."
2,SVM,0.993872,"{""classifier__kernel"": ""rbf"", ""classifier__gam..."
4,LightGBM,0.992097,"{""classifier__num_leaves"": 31, ""classifier__n_..."
3,XGBoost,0.991096,"{""classifier__subsample"": 1.0, ""classifier__n_..."
0,Logistic Regression,0.949803,"{""classifier__solver"": ""liblinear"", ""classifie..."



Saved tuned models: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction\models\tuned_models.joblib
Saved tuning results: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction\outputs\metrics\hyperparameter_tuning_results.csv
